In [1]:
import os

from pathlib import Path
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

print("Librerías de Langchain y Pinecone importadas correctamente.")

Librerías de Langchain y Pinecone importadas correctamente.


In [2]:

#* configuración de variables de entorno y rutas
load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

BASE_DIR = Path.cwd().parent.parent
DATA_DIR = BASE_DIR / "assets"
PDF_PATH = DATA_DIR / "terminosycondicionesbayerlatam.pdf"

print("Variables de entorno y rutas configuradas.")

Variables de entorno y rutas configuradas.


In [3]:

#* Configuración de indice y namespace en Pinecone
INDEX_NAME = "bayer-terms-and-conditions"
NAMESPACE = "bayer-latam"

In [4]:

#* Parámetros de fragmentación de documentos
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

In [5]:

#* Cargamos el PDF como lista de documentos
loader = PyPDFLoader(str(PDF_PATH))
docs_raw = loader.load()
print(f"Número de páginas cargadas: {len(docs_raw)}")

Número de páginas cargadas: 11


In [6]:

#* Definimos splitter para crear chunks manejables
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = CHUNK_SIZE,
    chunk_overlap = CHUNK_OVERLAP,
    separators = ["\n\n", "\n", ".", " ", ""]
)

In [7]:
docs = text_splitter.split_documents(docs_raw)
print(f"Número de chunks creados: {len(docs)}")
print(f"\nEjemplo de chunk (primeros 500 caracteres ):\n{docs[0].page_content[:500]}...")

Número de chunks creados: 96

Ejemplo de chunk (primeros 500 caracteres ):
Condiciones Generales de Compra 
 
1.- General.  
1.1.- Estas Condiciones Generales de Compra serán parte integral del Contrato/Orden de Compra  emitida por cualquier entidad 
del grupo Bayer en cualquier país de Latinoamérica, excepto, Argentina, Brasil, Bolivia, Chile, Paraguay and Uruguay.  El territorio 
latinoamericano norte en la que dicha entidad tenga su oficina corporativa principal  (en conjunto “Bayer”) , con relación a los 
servicios especializados no subordinados y/o el suministro y...


In [8]:

#* Inicializamos Pinecone
pinecone_client = Pinecone(api_key=PINECONE_API_KEY)

In [9]:

#* Revisamos si el índice ya existe en el proyecto
existing_indexes = [idx["name"] for idx in pinecone_client.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"Creando indice '{INDEX_NAME}' en Pinecone...")
    pinecone_client.create_index(
        name=INDEX_NAME,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )
    print("Índice creado.")
else:
    print(f"El índice '{INDEX_NAME}' ya existe en Pinecone.")

Creando indice 'bayer-terms-and-conditions' en Pinecone...
Índice creado.


In [10]:

#* Obtenemos el indice
pinecone_index = pinecone_client.Index(INDEX_NAME)
print(f"Índice '{INDEX_NAME}' obtenido correctamente.")

Índice 'bayer-terms-and-conditions' obtenido correctamente.


In [11]:

#* definimos el modelo de embeddings
embedding_model = OllamaEmbeddings(model="embeddinggemma:300m")
print("Modelo de embeddings Ollama configurado.")

Modelo de embeddings Ollama configurado.


In [12]:
docsearch = PineconeVectorStore.from_documents(
    documents=docs,
    embedding=embedding_model,
    index_name=INDEX_NAME,
    namespace=NAMESPACE
)

print("Documentos indexados correctamente en Pinecone.")

Index host ignored when initializing with index object.


Documentos indexados correctamente en Pinecone.


In [15]:

# Estadísticas del índice
stats = pinecone_index.describe_index_stats()

dimensions = stats.get("dimension", "N/A")
metric = stats.get("metric", "N/A")
total_vectors = stats.get("total_vector_count", 0)
vector_type = stats.get("vector_type", "N/A")
namespaces = stats.get("namespaces", {})

print(f"=== Resumen del índice '{INDEX_NAME}' ===")
print(f"Dimensiones: {dimensions}")
print(f"Métrica: {metric}")
print(f"Tipo de vector: {vector_type}")
print(f"Total de vectores: {total_vectors}")
print(f"Namespaces: {list(namespaces.keys())}")
print("\nDetalle por namespace:")
for ns, data in namespaces.items():
    count = data.get("vector_count", 0)
    print(f" -'{ns}': {count} vectores")

=== Resumen del índice 'bayer-terms-and-conditions' ===
Dimensiones: 768
Métrica: cosine
Tipo de vector: dense
Total de vectores: 96
Namespaces: ['bayer-latam']

Detalle por namespace:
 -'bayer-latam': 96 vectores


In [16]:

#* Cuando se requiera abrir de nuevo el indice lo hacemos con el siguiente código
"""
docsearch = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embedding_model,
    namespace=NAMESPACE
"""


#* Creación del retriever
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("Retriever creado a partir del índice Pinecone.")

Retriever creado a partir del índice Pinecone.


In [17]:
llm_model = ChatOllama(
    model="qwen3:4b",
    temperature=0.1
)

In [ ]:
template = """
Eres un asistente especializado en el Código de Conducta para Proveedores de Bayer.
Respondes SIEMPRE en español latinoamericano, de forma clara y profesional.

Usa exclusivamente la información del contexto para responder.
Si la pregunta no se puede responder con el contexto, di claramente 
que la información no está en el documento.


Pregunta del usuario:
{question}

Contexto:
{context}
"""

In [19]:
prompt = ChatPromptTemplate.from_template(template)

print("Modelo de chat y prompt RAG configurados.")

Modelo de chat y prompt RAG configurados.


In [20]:
rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever
    }
    | prompt
    | llm_model
)

print("Cadena RAG creada correctamente.")

Cadena RAG creada correctamente.


In [21]:
pregunta = (
    "Según el Código de Conducta para Proveedores de Bayer, "
    "¿Qué expectativas tiene Bayer sobre el comportamiento ético de sus proveedores?"
)

In [23]:
respuesta = rag_chain.invoke(pregunta)
print("\n=== Pregunta ===")
print(pregunta)
print("\n=== Respuesta ===")
print(respuesta.content)


=== Pregunta ===
Según el Código de Conducta para Proveedores de Bayer, ¿Qué expectativas tiene Bayer sobre el comportamiento ético de sus proveedores?

=== Respuesta ===
According to the provided context, Bayer expects its suppliers to:  

1. **Adhere to anti-corruption and anti-bribery policies** that align with Bayer’s standards and ensure these policies are communicated to all relevant employees, agents, subcontractors, and third parties involved in business transactions with Bayer.  
2. **Comply with applicable laws** related to transport, security, health, and labor.  
3. **Ensure ethical transaction practices**, meaning neither the supplier nor their representatives may authorize or offer transactions that violate the terms and conditions of the agreement.  

These requirements are explicitly outlined in Bayer’s Supplier Code of Conduct to maintain ethical standards and compliance across all supplier relationships.


In [25]:
print("\n--- Fragmentos utilizados ---")
fragmentos = retriever.invoke(pregunta)

for i, doc in enumerate(fragmentos, start=1):
    print(f"\nFragmento {i} - Página {doc.metadata.get('page', 'N/A')}:")
    print(doc.page_content[:400])
    print("-------------------------")


--- Fragmentos utilizados ---

Fragmento 1 - Página 8.0:
23.- Sostenibilidad. 
23.1- Se espera que la Contraparte organice su negocio con Bayer, en consonancia con el Código de Conducta de Proveedores de 
Bayer https://www.bayer.com/en/procurement/supplier-code-of-conduct). Bayer tendrá derecho a auditar el desempeño de 
sostenibilidad de la Contraparte, ya sea por una evaluación (online -en línea-, cuestionario de papel, etc.) o mediante una auditoría 
-------------------------

Fragmento 2 - Página 8.0:
transporte, seguridad, salud y laborales (en lo sucesivo conjuntamente denominadas como las “Leyes”) que sean aplicables a Bayer 
y la Contraparte, y a (ii) la Política Corporativa de Compliance de Bayer, la Política de Integridad de Bayer de México el Código de 
Conducta para Proveedores de Bayer (en lo sucesivo denominadas como las “Políticas Bayer”) mismas que están disponibles en 
www.bayer.mx
-------------------------

Fragmento 3 - Página 8.0:
de anticorrupción, mismos que son 

In [26]:

# * Se puede hacer limpieza del indice
pinecone_client.delete_index(name=INDEX_NAME)
print(f"Índice {INDEX_NAME} eliminado en Pinecone.")

Índice bayer-terms-and-conditions eliminado en Pinecone.
